# A-FIS Regression — Application

**Purpose:** Train and deploy an A-FIS regression model.

1. **Setup & Configure** - Dataset and model parameters
2. **K-Fold Selection** - Use K-fold to select the best model
3. **Test & Visualize** - Evaluate on held-out test set
4. **Save Model** - Persist for practical use

---

**Note:** This is for MODEL DEPLOYMENT, not method benchmarking.
For robust method evaluation (RMSE ± std), see `afis_regression_benchmark.ipynb`.


---
## 1. Setup & Configuration


In [2]:
import os
import sys
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split

# Path setup - add AFIS_Repo root to path
NOTEBOOK_DIR = os.getcwd()
REPO_ROOT = os.path.dirname(os.path.dirname(os.path.dirname(NOTEBOOK_DIR)))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Import from afis package
from afis.regression import (
    AFISRegressor, 
    generate_rule_base, 
    evaluate_kfold,
    compute_metrics, 
    print_metrics,
    plot_predictions_vs_actual
)

# Local dataset config (still in experiments/)
from datasets_config import get_dataset_path, get_dataset_config

print("✓ Modules loaded")


✓ Modules loaded


/Users/rpereiratractian.com/PhD_Scripts/Ph_D_scripts/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
# =============================================================================
# CONFIGURATION
# =============================================================================

# Dataset selection
SELECTED_DATASET = 'chemical_conc'

# Load dataset config
dataset_config = get_dataset_config(SELECTED_DATASET)
N_FUZZY_PARTITIONS = dataset_config['n_fuzzy_part_dim']
N_RULES = dataset_config['n_rules']
FILTER_METHOD = 'balanced'  # 'balanced' = N per consequent, 'top_n' = top N by strength

# Load data
df = pd.read_pickle(get_dataset_path(SELECTED_DATASET))
if dataset_config['skip_rows'] > 0:
    df = df.iloc[dataset_config['skip_rows']:]
df = df.dropna()

# Model configuration
# Note: ['weighted_avg', []] signals that correlation weights should be
# computed per-fold on each fold's training data (not pre-computed)
# Use 'k_fixed' for fixed k (no optimization), or 'k_max' to search for optimal k
MODEL_CONFIG = {
    'agg_method': ['weighted_avg', []],  # Weights computed per-fold
    't_norm_type': 'luka',
    'imp_params': ['dombi', 1],  # Parametric implication
    'disc': 100,
    'k_max': 10,  # Fixed k (no optimization) — use 'k_max' instead for k search
    'p_norm': 1,
    'param_range': (0.1, 50.0),
    'param_step': 0.5,
}

# Train/test split
N_FOLDS = 5           # K in K-fold for model selection
TEST_SIZE = 0.2
RANDOM_STATE = 42

# Display configuration
k_info = f"k_fixed={MODEL_CONFIG['k_fixed']}" if 'k_fixed' in MODEL_CONFIG else f"k_max={MODEL_CONFIG['k_max']}"
print(f"Dataset: {SELECTED_DATASET.upper()}")
print(f"Shape: {df.shape}")
print(f"Fuzzy partitions: {N_FUZZY_PARTITIONS}, Target rules: {N_RULES}, Filter: {FILTER_METHOD}")
print(f"\nModel: imp={MODEL_CONFIG['imp_params'][0]}, {k_info}")


Dataset: CHEMICAL_CONC
Shape: (194, 4)
Fuzzy partitions: 6, Target rules: 21, Filter: balanced

Model: imp=dombi, k_max=10


In [13]:
df

,sample0,sample1,sample2,sample3
0,17.0,16.6,16.3,16.1
1,16.6,16.3,16.1,17.1
2,16.3,16.1,17.1,16.9
3,16.1,17.1,16.9,16.8
4,17.1,16.9,16.8,17.4
...,...,...,...,...
189,17.0,18.0,18.2,17.6
190,18.0,18.2,17.6,17.8
191,18.2,17.6,17.8,17.7
192,17.6,17.8,17.7,17.2


In [14]:
# plot the i-th column of df



---
## 2. K-Fold Model Selection

Use K-fold CV to select the best model. Each fold trains a complete model (rule associations + parameter optimization + k-search), and the best one is selected.


In [15]:
# Prepare data
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].to_numpy()

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
print(f"Train+Val: {len(X_trainval)}, Test: {len(X_test)}")

# Rule base generator function (verbose=True shows rule stats per fold)
def make_rule_base(X_train, y_train):
    return generate_rule_base(X_train, y_train, N_FUZZY_PARTITIONS, N_RULES,
                              filter_method=FILTER_METHOD, verbose=True)

# Run K-fold for model selection
cv_results = evaluate_kfold(
    X_trainval, y_trainval,
    rule_base_generator=make_rule_base,
    config=MODEL_CONFIG,
    n_splits=N_FOLDS,
    random_state=RANDOM_STATE
)


Train+Val: 155, Test: 39

Fold 1/5
  Correlation weights: ['0.278', '0.321', '0.402']
  After cleaning (unique antecedents): 21 rules
    By consequent: {'D2': 1, 'D3': 12, 'D4': 7, 'D5': 1}
  After filtering (max 7/consequent): 16 rules
    By consequent: {'D2': 1, 'D3': 7, 'D4': 7, 'D5': 1}
  Rules: 16
[1/3] Building point-rule associations...


Building associations: 100%|██████████| 124/124 [00:00<00:00, 662.29it/s]


[2/3] Optimizing implication parameters...


Optimizing params: 100%|██████████| 124/124 [00:03<00:00, 38.53it/s]


    Param stats: mean=10.16, std=15.69
[3/3] Finding optimal k...


...

[2/3] Optimizing implication parameters...


Optimizing params: 100%|██████████| 124/124 [00:03<00:00, 37.80it/s]


    Param stats: mean=12.40, std=18.40
[3/3] Finding optimal k...


Optimizing k: 100%|██████████| 31/31 [00:00<00:00, 231.76it/s]


    Optimal k: 8
  Validation RMSE: 0.2872

K-Fold Summary
Mean RMSE: 0.3350 ± 0.0439
Best fold: 3 (RMSE=0.2848)


---
## 3. Select Best Model

The best model from K-fold (lowest validation RMSE) is used directly for testing.
No retraining is performed — this matches the original notebook behavior.


In [16]:
# Use best model from K-fold directly (no retraining)
best_model = cv_results['best_model']

# Find which fold was best
all_results = cv_results['all_results']
best_fold_idx = min(range(len(all_results)), 
                    key=lambda i: all_results[i]['val_rmse'])
best_fold = all_results[best_fold_idx]

print(f"Best Model (from Fold {best_fold['fold']}):")
print(f"  Validation RMSE: {best_fold['val_rmse']:.4f}")
print(f"  Optimal k: {best_fold['optimal_k']}")
print(f"  Rules: {best_fold['n_rules']}")


Best Model (from Fold 3):
  Validation RMSE: 0.2848
  Optimal k: 8
  Rules: 17


---
## 4. Test Evaluation & Visualization


In [17]:
# Make predictions on test set using best model from K-fold
test_predictions = best_model.predict(X_test)

# Compute and display metrics
print("\n" + "="*50)
print("TEST RESULTS")
print("="*50)
print_metrics(y_test, test_predictions)

# Visualization
fig = plot_predictions_vs_actual(
    y_test, test_predictions,
    title=f'A-FIS Regression: {SELECTED_DATASET.upper()} (k={best_model.optimal_k})'
)
fig.show()

# Summary comparison
print("\n" + "="*50)
print("SUMMARY")
print("="*50)
print(f"CV Mean RMSE:  {cv_results['mean_rmse']:.4f} ± {cv_results['std_rmse']:.4f}")
print(f"Test RMSE:     {compute_metrics(y_test, test_predictions)['rmse']:.4f}")
print(f"Optimal k:     {best_model.optimal_k}")


Predicting: 100%|██████████| 39/39 [00:00<00:00, 303.38it/s]


TEST RESULTS
  RMSE: 0.2938
  MAE:  0.2284
  R²:   0.2239



SUMMARY
CV Mean RMSE:  0.3350 ± 0.0439
Test RMSE:     0.2938
Optimal k:     8


In [18]:
# Compare test vs prediction (time series order)
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(y=y_test, mode='lines', name='Actual', line=dict(color='blue')))
fig.add_trace(go.Scatter(y=test_predictions, mode='lines', name='Prediction', line=dict(color='red', dash='dash')))

fig.update_layout(
    title=f'Test vs Prediction -  (RMSE={compute_metrics(y_test, test_predictions)["rmse"]:.4f})',
    xaxis_title='Sample Index',
    yaxis_title='Value',
    legend=dict(x=0.01, y=0.99),
    height=500
)
fig.show() 




---
## 5. Save Model (Optional)

Save the trained model for later use.


In [19]:
# Uncomment to save
# best_model.save(f'afis_model_{SELECTED_DATASET}.pkl')

# To load later:
# model = AFISRegressor.load(f'afis_model_{SELECTED_DATASET}.pkl')
# predictions = model.predict(new_data)
